New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import sys, os, inspect
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
from ai_data_scientist import project_manager, lifecycle, language_router
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-19-geyser')
project_manager.ensure_notebook(handle)
project_manager.ensure_data_dir(handle)
lifecycle.register_run('v020-exp-19', handle.notebook_path)
print('language:', language_router.detect_language('噴出時間と待ち時間の関係を分析してください'))
print('notebook:', handle.notebook_path)
print('registered:', lifecycle.get_run_status('v020-exp-19'))
import kaggle, nbformat
from ai_data_scientist import ingestion, cleaning, eda, stats_analysis, data_quality, notebook_audit
for module, names in [(ingestion,['ingest']), (cleaning,['clean_dataset']), (eda,['explore']), (stats_analysis,['correlation']), (data_quality,['detect_anomalies','validate_anomalies']), (notebook_audit,['audit_notebook','audit_visual_outputs'])]:
    for name in names:
        fn = getattr(module,name)
        print(module.__name__,name,inspect.signature(fn))
print(inspect.getsource(notebook_audit.audit_visual_outputs))

language: ja
notebook: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-19-geyser/notebooks/kaggle-v020-kaggle-exp-19-geyser.ipynb
registered: RunStatus(run_id='v020-exp-19', state='running', active_cell_executions=0, pending_notebook_writes=0, locks_held=0, notebook_path='/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-19-geyser/notebooks/kaggle-v020-kaggle-exp-19-geyser.ipynb', reason=None)
ai_data_scientist.ingestion ingest (source_spec: 'SourceSpec', fetcher=None, allowlist: 'tuple[str, ...]' = (), row_limit: 'int' = 100000) -> 'IngestionResult'
ai_data_scientist.cleaning clean_dataset (df: 'pd.DataFrame', operation: 'str', columns: 'list | None' = None, fill_value=None) -> 'CleaningReport'
ai_data_scientist.eda explore (df: 'pd.DataFrame', top_n: 'int' = 10) -> 'EDAReport'
ai_data_scientist.stats_analysis correlation (df: 'pd.DataFrame', col_a: 'str', col_b: 'str', language: 'str' = 'en') -> 'StatResult'
ai_data_scientist.data

In [2]:
import nbformat
setup = '''from pathlib import Path
import os, sys, json, hashlib, io, contextlib, re, warnings, zipfile
from datetime import datetime, timezone
from dataclasses import asdict
from contextlib import contextmanager
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
import numpy as np
import pandas as pd
import scipy.stats as st
import requests
import matplotlib.pyplot as plt
import nbformat
from IPython.display import display
from kaggle.api.kaggle_api_extended import KaggleApi
from ai_data_scientist import project_manager, lifecycle, language_router, ingestion, cleaning, eda, stats_analysis, visualization, data_definition, analysis_assumptions, data_quality, sensitivity, insight_engine, notebook_audit
RUN_ID = 'v020-exp-19'
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-19-geyser')
project_manager.ensure_notebook(handle)
data_dir = project_manager.ensure_data_dir(handle)
lifecycle.register_run(RUN_ID, handle.notebook_path)
phase_log = []
def log_event(phase, event):
    phase_log.append({'utc': datetime.now(timezone.utc).isoformat(), 'phase': phase, 'event': event})
@contextmanager
def phase(name):
    if lifecycle.is_cancel_requested(RUN_ID):
        raise RuntimeError('実行中止が要求されました')
    lifecycle.mark_execution_start(RUN_ID)
    log_event(name, 'execution_start')
    try:
        yield
    except Exception:
        lifecycle.mark_failed(RUN_ID)
        log_event(name, 'failed')
        raise
    finally:
        lifecycle.mark_execution_end(RUN_ID)
        log_event(name, 'execution_end')
def queued_write(mutation):
    lifecycle.mark_write_start(RUN_ID)
    log_event('notebook', 'write_start')
    try:
        return project_manager.enqueue_write(handle, mutation)
    finally:
        lifecycle.mark_write_end(RUN_ID)
        log_event('notebook', 'write_end')
print('言語:', language_router.detect_language('噴出時間と待ち時間を分析する'))
print('run_id:', RUN_ID)
print('ライフサイクル:', asdict(lifecycle.get_run_status(RUN_ID)))
'''
acquire = '''with phase('Kaggle検索・取得'):
    search_log = []
    api_failures = []
    selected = None
    def safe_failure(stage, exc):
        response = getattr(exc, 'response', None)
        status = getattr(exc, 'status', None) or getattr(response, 'status_code', None)
        item = {'stage': stage, 'exception_class': type(exc).__name__, 'http_status': status,
                'detail': '認証情報の漏洩防止のため例外本文とSDK生ログは保存しない'}
        api_failures.append(item)
        print('API失敗:', json.dumps(item, ensure_ascii=False))
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api = KaggleApi()
            api.authenticate()
        print('KaggleApi().authenticate(): 成功（認証情報非表示）')
        candidates = []
        for query in ['old faithful geyser', 'old faithful', 'geyser']:
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    found = api.dataset_list(search=query, page_size=10)
                rows = [{'ref': str(d.ref), 'title': str(d.title)} for d in found]
                search_log.append({'query': query, 'datasets': rows})
                print('dataset_list:', json.dumps(search_log[-1], ensure_ascii=False))
                candidates.extend(rows)
            except Exception as exc:
                safe_failure('dataset_list:'+query, exc)
        seen = set()
        for candidate in candidates:
            ref = candidate['ref']
            if ref in seen:
                continue
            seen.add(ref)
            if not any(term in (ref + ' ' + candidate['title']).lower() for term in ['faithful', 'geyser']):
                continue
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    listing = api.dataset_list_files(ref)
                files = [{'name': f.name, 'bytes': getattr(f, 'total_bytes', None)} for f in listing.files]
                print('dataset_list_files:', ref, json.dumps(files, ensure_ascii=False))
                search_log.append({'ref': ref, 'files': files})
                for entry in files:
                    name = entry['name']
                    if not name.lower().endswith('.csv'):
                        continue
                    if entry['bytes'] is not None and entry['bytes'] > 10_000_000:
                        continue
                    stage_dir = data_dir / ref.replace('/', '--')
                    stage_dir.mkdir(parents=True, exist_ok=True)
                    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                        api.dataset_download_file(ref, name, path=str(stage_dir), quiet=True, force=True)
                    local = stage_dir / Path(name).name
                    if not local.exists():
                        archives = list(stage_dir.glob('*.zip'))
                        for archive in archives:
                            with zipfile.ZipFile(archive) as zf:
                                match = [m for m in zf.namelist() if Path(m).name == Path(name).name]
                                if match:
                                    local.write_bytes(zf.read(match[0]))
                                    break
                    if not local.exists():
                        raise FileNotFoundError('SDKダウンロード後に期待するCSVが存在しない')
                    probe = pd.read_csv(local)
                    normalized = {c: str(c).strip().lower() for c in probe.columns}
                    duration_col = next((c for c, v in normalized.items() if v in ['duration', 'eruptions', 'eruption', 'eruptions_min', 'eruption time']), None)
                    waiting_col = next((c for c, v in normalized.items() if v in ['waiting', 'waiting time', 'waiting_min']), None)
                    if duration_col and waiting_col:
                        selected = {'kind': 'kaggle', 'owner_dataset_slug': ref, 'filename': name,
                                    'local_path': str(local), 'duration_column': duration_col, 'waiting_column': waiting_col,
                                    'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
                                    'sha256': hashlib.sha256(local.read_bytes()).hexdigest(),
                                    'dataset_url': 'https://www.kaggle.com/datasets/'+ref}
                        break
                    print('対象外CSV:', ref, name, 'columns=', list(probe.columns))
                if selected:
                    break
            except Exception as exc:
                safe_failure('dataset_list_files/download:'+ref, exc)
    except Exception as exc:
        safe_failure('authenticate', exc)
    if selected is None:
        print('Kaggle API検索・取得失敗に限ったフォールバックを実施')
        legacy = Path('/home/nahisaho/kaggle/experiments/white-paper.md')
        if legacy.exists():
            legacy_text = legacy.read_text(encoding='utf-8')
            print('旧実験資料の指定URL確認:', 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/geyser.csv' in legacy_text)
        url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/geyser.csv'
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        local = data_dir / 'geyser-fallback.csv'
        local.write_bytes(response.content)
        selected = {'kind': 'fallback', 'owner_dataset_slug': None, 'filename': local.name, 'local_path': str(local),
                    'duration_column': 'duration', 'waiting_column': 'waiting', 'csv_url': url,
                    'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
                    'sha256': hashlib.sha256(response.content).hexdigest(),
                    'reason': 'Kaggle APIで対応データを検索・取得できなかった',
                    'identity_caveat': '公開CSVとKaggle掲載版の完全な同一性は保証されない'}
    provenance = {'selected': selected, 'search_log': search_log, 'api_failures': api_failures}
    (data_dir / 'provenance.json').write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding='utf-8')
    print('取得記録:', json.dumps(selected, ensure_ascii=False, indent=2))
    loaded = ingestion.ingest(ingestion.SourceSpec('csv', selected['local_path']), row_limit=10000)
    if loaded.truncated:
        raise ValueError('行数上限による切捨てが発生したため分析を停止')
    raw_df = loaded.dataframe
    df = raw_df.rename(columns={selected['duration_column']: 'duration', selected['waiting_column']: 'waiting'})[['duration','waiting']].apply(pd.to_numeric, errors='raise')
    print('shape:', raw_df.shape, 'columns:', list(raw_df.columns))
    display(raw_df.head())
    print('検証用SHA256:', hashlib.sha256(Path(selected['local_path']).read_bytes()).hexdigest())
'''
lifecycle.mark_write_start('v020-exp-19')
def initial(nb):
    if nb.cells:
        raise RuntimeError('保存先は既存の非空Notebookです。上書きしません。')
    nb.cells.extend([
        nbformat.v4.new_markdown_cell('# Old Faithful間欠泉：噴出時間と待ち時間\n\nrun_id: `v020-exp-19`。AI Data Scientist Skill / Jupyter MCPで実行。\n\n## 分析計画\n1. Kaggle APIで公開データの候補・ファイルを調べ、対応CSVを取得し由来・日時・SHA-256を保存する。\n2. 単位・定義・観測範囲の確度を区別し、欠損・重複・意味的な異常を調べる。\n3. ヒストグラム/KDE、ECDF、散布図と条件別分布で二峰性と全体相関の内訳を調べる。\n4. 群内相関・共分散分解・境界感度を調べる。予測モデルの交差検証・不確実性は結果を読み直して必要時に追加する。\n5. 最大3回の追加依頼、直接利用モジュール、改善候補を記録し、クリーンなカーネルで全コードセルを順番に再実行する。\n\n因果推論や現代の現地運用予測を目的にしない。観測順・時刻・待ち時間の対応方向は出典確認を優先し、不明なら留保する。'),
        nbformat.v4.new_code_cell(setup, id='setup'),
        nbformat.v4.new_markdown_cell('## Kaggle公開データの検索・取得\nSDKの標準出力・標準エラーを捕捉し、認証情報・生例外本文は出力しない。API失敗は例外型、HTTP状態、実施段階を記録する。Kaggle参照先: https://www.kaggle.com/datasets?search=old+faithful+geyser 。'),
        nbformat.v4.new_code_cell(acquire, id='acquire')])
project_manager.enqueue_write(handle, initial)
lifecycle.mark_write_end('v020-exp-19')
print('初期セルを書込み:', handle.notebook_path)

初期セルを書込み: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-19-geyser/notebooks/kaggle-v020-kaggle-exp-19-geyser.ipynb


In [3]:
lifecycle.mark_write_start('v020-exp-19')
def repair_sdk(nb):
    cell = next(c for c in nb.cells if c.id == 'acquire')
    cell.source = cell.source.replace('api.dataset_list(search=query, page_size=10)', 'api.dataset_list(search=query)')
    nb.cells.insert(2, nbformat.v4.new_markdown_cell('## 実行環境の問題と解決ログ\n初回取得セル（execution_count=2）で `dataset_list(search=..., page_size=10)` が TypeError。インストール済SDKのシグネチャをJupyter MCPで確認し、未対応の `page_size` 引数を削除して再試行した。認証は成功しており、認証障害ではない。初回のみ暫定フォールバックCSVを取得したが、最終的な採用元は再試行セルの取得記録で決定する。この呼出し誤りは分析コード/Kaggle SDKのインターフェースに関するもので、Jupytermindの不具合とは分類しない。例外本文・認証値は保存しない。', id='sdk-log'))
project_manager.enqueue_write(handle, repair_sdk)
lifecycle.mark_write_end('v020-exp-19')
print('SDK引数を修正。再検索を実施する。')

SDK引数を修正。再検索を実施する。


In [4]:
definitions_code = '''with phase('定義・前提・品質検査'):
    import html
    dictionary_url = 'https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/faithful.html'
    dictionary_verified = False
    try:
        dictionary_response = requests.get(dictionary_url, timeout=30)
        dictionary_response.raise_for_status()
        dictionary_text = html.unescape(re.sub('<[^>]+>', ' ', dictionary_response.text))
        dictionary_text = re.sub(r'\\s+', ' ', dictionary_text).strip()
        (data_dir / 'faithful-dictionary.html').write_bytes(dictionary_response.content)
        dictionary_verified = 'eruptions' in dictionary_text and 'waiting' in dictionary_text and 'minutes' in dictionary_text
        print('Rデータ辞書:', dictionary_url)
        print('定義・注意書き:', dictionary_text[:4500])
    except requests.RequestException as exc:
        print('辞書取得失敗:', type(exc).__name__, '単位の確度をinferredとする')
    F = data_definition.FieldValue
    definitions = data_definition.build_manifest(
        source={'file_identity': F(selected, 'verified', '取得セルのSDK結果とSHA-256'),
                'semantic_dictionary': F(dictionary_url, 'verified' if dictionary_verified else 'unknown', dictionary_url),
                'original_measurement_protocol': F(None, 'unknown', 'CSVには測定手順がない')},
        dataset_scope={'site': F('Old Faithful, Yellowstone National Park, Wyoming, USA', 'verified' if dictionary_verified else 'inferred', dictionary_url),
                       'row_count': F(len(df), 'verified', 'CSV読込み'),
                       'observation_dates': F(None, 'unknown', 'CSVに時刻なし'),
                       'chronological_order': F(None, 'unknown', '連番は順序の証明ではない'),
                       'current_population_representativeness': F(None, 'unknown', '歴史的標本・抽出設計不明')},
        variables={'duration': {'unit': F('minutes', 'verified' if dictionary_verified else 'inferred', dictionary_url),
                                'definition': F('噴出の継続時間', 'verified' if dictionary_verified else 'inferred', dictionary_url)},
                   'waiting': {'unit': F('minutes', 'verified' if dictionary_verified else 'inferred', dictionary_url),
                               'definition': F('噴出間の待ち時間', 'verified' if dictionary_verified else 'inferred', dictionary_url),
                               'paired_direction': F('同じ行の噴出後に次の噴出までか、直前の間隔かは辞書だけでは未確認', 'unknown', dictionary_url),
                               'start_end_convention': F(None, 'unknown', '開始間隔か終了後間隔かは未確認')}}},
        transformations=('eruptionsをdurationへ改名', '連番列を分析変数から除外', 'CSV行順を保持', '重複観測ペアを削除しない'))
    print('データ定義manifest:', json.dumps(asdict(definitions), ensure_ascii=False, indent=2))
    print('unresolved_fields():', [(k, asdict(v)) for k,v in definitions.unresolved_fields()])
    inferred = [(scope+'.'+key, asdict(value)) for scope, values in [('source', definitions.source), ('dataset_scope', definitions.dataset_scope)] for key,value in values.items() if value.status == 'inferred']
    inferred += [(f'variables.{name}.{key}', asdict(value)) for name,values in definitions.variables.items() for key,value in values.items() if value.status == 'inferred']
    print('inferredの留保:', inferred)
    schema = {'duration': {'min': 0.01, 'max': 10, 'not_null': True}, 'waiting': {'min': 1, 'max': 180, 'not_null': True}}
    semantic_anomalies = data_quality.detect_anomalies(df, schema=schema)
    print('意味的異常（広い物理的妥当域、公式基準ではない）:', [asdict(a) for a in semantic_anomalies])
    print('欠損:', df.isna().sum().to_dict(), '無限値:', int(np.isinf(df.to_numpy()).sum()))
    duplicate_pairs = int(df.duplicated().sum())
    print('同一(duration, waiting)ペアの追加出現数:', duplicate_pairs, '丸め・反復観測の可能性があるので削除しない')
    cleaning_report = cleaning.clean_dataset(df, operation='drop_na', columns=['duration','waiting'])
    df = cleaning_report.dataframe
    print('clean_dataset:', {k:v for k,v in asdict(cleaning_report).items() if k != 'dataframe'})
    if len(semantic_anomalies) or np.isinf(df.to_numpy()).any():
        raise ValueError('意味的異常・非有限値は自動修正せず原因の確認が必要')
    assumptions = analysis_assumptions.AnalysisAssumptionManifest(
        analysis_scope={'target': '取得CSVの記述・関連分析', 'threshold_minutes': 3.0, 'not_target': '因果効果・現在の現地運用予測'},
        assumptions=(analysis_assumptions.Assumption('paired-direction', 'waitingが噴出後の次回待ち時間と対応する', 'assumed', conclusion_critical=True, impact_if_false='次回予測という解釈は成立しない。行内の関連だけを報告する'),
                     analysis_assumptions.Assumption('row-independence', '行が独立同分布で、再標本化・ランダムCVが将来予測誤差を表す', 'assumed', conclusion_critical=True, impact_if_false='区間・CVは参考値。順序分割・ブロック再標本化を追加検討する'),
                     analysis_assumptions.Assumption('duplicates-preserved', '同じ値の観測を別イベントとして保持する', 'assumed', conclusion_critical=False, impact_if_false='重複除去感度を確認する'),
                     analysis_assumptions.Assumption('threshold', '3分を探索的な短長境界とする', 'assumed', conclusion_critical=False, impact_if_false='2.5–3.5分の境界感度で評価する')),
        causal_scope='associational', sampling={'n': len(df), 'seed': 20261002})
    assumption_findings = analysis_assumptions.check_manifest(assumptions)
    print('分析前提manifest:', json.dumps(asdict(assumptions), ensure_ascii=False, indent=2))
    print('check_manifest findings:', [asdict(f) for f in assumption_findings])
    print('適用外: 独立参照データがないためvalidate_anomalies/compare_datasetsは未実施。別コピーを独立標本とみなさない。因果推論、季節性、時間単位の周期分析は時刻・識別情報不足。クラスタリング/AutoMLはMVP外であり実施せず、閾値・KDEで記述する。')
    (data_dir / 'data-definition-manifest.json').write_text(json.dumps(asdict(definitions), ensure_ascii=False, indent=2), encoding='utf-8')
    (data_dir / 'analysis-assumptions-manifest.json').write_text(json.dumps(asdict(assumptions), ensure_ascii=False, indent=2), encoding='utf-8')
'''
summary_code = '''with phase('統計要約・分布・群間構造'):
    from scipy.signal import find_peaks
    report = eda.explore(df)
    print('eda.explore:', report)
    display(df.describe(percentiles=[.05,.25,.5,.75,.95]).T.round(4))
    corr = stats_analysis.correlation(df, 'duration', 'waiting', language='ja')
    print('Jupytermind相関:', corr)
    print('interpretation:', corr.interpretation)
    pearson = st.pearsonr(df.duration, df.waiting)
    spearman = st.spearmanr(df.duration, df.waiting)
    print(f'PEARSON_R={pearson.statistic:.6f}; SPEARMAN_R={spearman.statistic:.6f}')
    group = np.where(df.duration < 3.0, 'short', 'long')
    analysis_df = df.assign(group=group)
    groups = analysis_df.groupby('group', sort=True).agg(n=('waiting','size'), duration_mean=('duration','mean'), duration_median=('duration','median'), duration_sd=('duration','std'), waiting_mean=('waiting','mean'), waiting_median=('waiting','median'), waiting_sd=('waiting','std'), waiting_min=('waiting','min'), waiting_max=('waiting','max'))
    display(groups.round(4))
    within_corr = {g: float(st.pearsonr(sub.duration, sub.waiting).statistic) for g,sub in analysis_df.groupby('group')}
    print('群内Pearson:', within_corr)
    x, y = df.duration.to_numpy(), df.waiting.to_numpy()
    total_cov = float(np.mean((x-x.mean())*(y-y.mean())))
    within_cov = sum(len(sub)/len(df)*np.mean((sub.duration-sub.duration.mean())*(sub.waiting-sub.waiting.mean())) for _,sub in analysis_df.groupby('group'))
    between_cov = total_cov - within_cov
    between_fraction = between_cov/total_cov
    wait_gap = float(groups.loc['long','waiting_mean'] - groups.loc['short','waiting_mean'])
    print(f'WAIT_GAP={wait_gap:.6f}; BETWEEN_COV_FRACTION={between_fraction:.6f}')
    print('共分散分解（ddof=0）:', {'total':total_cov, 'within':float(within_cov),'between':float(between_cov)})
    print('注意: 共分散割合は因果寄与率や説明分散R2ではない。durationで群を切るため群内相関の低下には範囲制限も含む。')
    kde_modes = []
    for column in ['duration','waiting']:
        values = df[column].to_numpy()
        grid = np.linspace(values.min() - values.std()*.15, values.max()+values.std()*.15, 1200)
        for multiplier in [.5,.75,1.,1.25]:
            kde = st.gaussian_kde(values)
            kde.set_bandwidth(kde.scotts_factor()*multiplier)
            density = kde(grid)
            peaks,_ = find_peaks(density, prominence=density.max()*.05)
            kde_modes.append({'column': column, 'Scott_multiplier': multiplier, 'n_peaks':len(peaks), 'peak_locations': [round(float(grid[p]),4) for p in peaks]})
    print('KDE帯域感度（有意性検定ではない）:', json.dumps(kde_modes, ensure_ascii=False))
    def separation(threshold, deduplicate):
        sample = df.drop_duplicates() if deduplicate else df
        means = sample.assign(group=np.where(sample.duration < threshold,'short','long')).groupby('group').waiting.mean()
        return float(means['long']-means['short'])
    sensitivity_report = sensitivity.run_sensitivity(sensitivity.SensitivityPlan({'threshold':[3.0,2.5,2.75,3.25,3.5], 'deduplicate':[False,True]}, max_runs=10), separation, stability_tolerance=.10)
    print('感度分析:', json.dumps(asdict(sensitivity_report), ensure_ascii=False))
    print(f'SENSITIVITY_STABLE={sensitivity_report.stable}; MAX_RELATIVE_DEVIATION={sensitivity_report.max_relative_deviation:.6f}')
    first_results = {'pearson':float(pearson.statistic),'spearman':float(spearman.statistic),'groups':groups.to_dict(),'within_correlation':within_corr,'waiting_gap':wait_gap,'between_cov_fraction':float(between_fraction), 'kde_modes':kde_modes,'sensitivity':asdict(sensitivity_report)}
    (data_dir/'initial-results.json').write_text(json.dumps(first_results, ensure_ascii=False, indent=2), encoding='utf-8')
'''
charts_code = '''with phase('初回図表'):
    chart_checks = []
    def emit_png(png_bytes, chart_name, captured):
        glyph_warnings = [str(w.message) for w in captured if 'Glyph' in str(w.message)]
        chart_checks.append({'chart':chart_name,'missing_glyphs':glyph_warnings,'png_bytes':len(png_bytes)})
        if glyph_warnings:
            raise ValueError('図表の欠落グリフを検出: '+str(glyph_warnings))
        bundle = visualization.build_image_output(png_bytes)
        display(dict(bundle.data), raw=True)
    def emit_figure(fig, chart_name):
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter('always')
            buffer = io.BytesIO()
            fig.savefig(buffer, format='png', dpi=120, bbox_inches='tight')
        plt.close(fig)
        emit_png(buffer.getvalue(), chart_name, captured)
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always')
        png = visualization.render_chart(df, kind='scatter', x='duration', y='waiting', title='Old Faithful：噴出時間と待ち時間', xlabel='噴出時間（分）', ylabel='噴出間の待ち時間（分）')
    emit_png(png, '日本語散布図', captured)
    colors = {'short':'#2274A5','long':'#D1495B'}
    fig, axes = plt.subplots(2, 2, figsize=(11,8))
    for j,col in enumerate(['duration','waiting']):
        values = df[col].to_numpy()
        axes[0,j].hist(values,bins=18,density=True,alpha=.4,color='#555555', label='Histogram')
        grid = np.linspace(values.min()-.2*values.std(),values.max()+.2*values.std(),500)
        axes[0,j].plot(grid,st.gaussian_kde(values)(grid),label='KDE (Scott)')
        axes[0,j].set(title=f'{col}: histogram and KDE',xlabel=f'{col} (min)',ylabel='Density')
        axes[0,j].legend()
        for g,sub in analysis_df.groupby('group'):
            ordered = np.sort(sub[col])
            axes[1,j].step(ordered,np.arange(1,len(ordered)+1)/len(ordered),where='post',color=colors[g],label=f'{g}, n={len(sub)}')
        axes[1,j].set(title=f'{col}: group ECDF',xlabel=f'{col} (min)',ylabel='Cumulative proportion')
        axes[1,j].legend()
    fig.tight_layout()
    emit_figure(fig, '周辺分布と群別ECDF')
    fig,axes = plt.subplots(1,2,figsize=(11,4))
    for g,sub in analysis_df.groupby('group'):
        axes[0].scatter(sub.duration,sub.waiting,s=20,alpha=.65,color=colors[g],label=f'{g}: r={within_corr[g]:.3f}')
    slope,intercept = np.polyfit(df.duration,df.waiting,1)
    grid=np.linspace(df.duration.min(),df.duration.max(),200)
    axes[0].plot(grid,intercept+slope*grid,color='black',ls='--',label='Pooled OLS')
    axes[0].axvline(3,color='grey',ls=':',label='Exploratory 3-min threshold')
    axes[0].set(title='Two regimes and pooled regression',xlabel='Duration (min)',ylabel='Waiting (min)')
    axes[0].legend(fontsize=8)
    axes[1].boxplot([analysis_df.loc[analysis_df.group==g,'waiting'] for g in ['short','long']],tick_labels=['short','long'],showmeans=True)
    rng=np.random.default_rng(20261002)
    for i,g in enumerate(['short','long'],start=1):
        vals=analysis_df.loc[analysis_df.group==g,'waiting'].to_numpy()
        axes[1].scatter(i+rng.uniform(-.16,.16,len(vals)),vals,color=colors[g],s=9,alpha=.4,label=f'{g} observations')
    axes[1].set(title='Conditional waiting distributions',xlabel='Duration group (threshold 3 min)',ylabel='Waiting (min)')
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    emit_figure(fig, '群構造と条件別待ち時間')
    print('図表描画ログ:', json.dumps(chart_checks, ensure_ascii=False))
'''
lifecycle.mark_write_start('v020-exp-19')
def append_first(nb):
    nb.cells.extend([
        nbformat.v4.new_markdown_cell('## データ定義・分析前提manifestと意味的異常検査\nR公式データ辞書は単位・意味の参考元であり、Kaggleの元データとの同一性検証ではない。次回への方向対応、測定時刻、現代への代表性は別に留保する。',id='definitions-note'),
        nbformat.v4.new_code_cell(definitions_code,id='definitions'),
        nbformat.v4.new_markdown_cell('## 初回分析：分布・統計的要約・群間と群内の関係\n3分未満をshort、3分以上をlongとする探索的な区分。外れ値の機械的除去や既存kind列の採用は行わない。KDEの峰は帯域に依存するため複数帯域を検討し、二峰性の正式検定とは区別する。',id='initial-note'),
        nbformat.v4.new_code_cell(summary_code,id='summary'),
        nbformat.v4.new_code_cell(charts_code,id='initial-charts',metadata={'chart':{'title':'散布図・周辺分布・ECDF・条件別分布','xlabel':'Duration / Waiting (min) / Duration group','ylabel':'Waiting (min) / Density / Cumulative proportion','legend':'short / long / KDE / histogram / pooled OLS','missing_glyphs':[]}})])
project_manager.enqueue_write(handle,append_first)
lifecycle.mark_write_end('v020-exp-19')
print('定義・初回分析・図表セルを記録')

定義・初回分析・図表セルを記録


In [5]:
lifecycle.mark_write_start('v020-exp-19')
def fix_definition(nb):
    c = next(c for c in nb.cells if c.id == 'definitions')
    c.source = c.source.replace("'開始間隔か終了後間隔かは未確認')}}},", "'開始間隔か終了後間隔かは未確認')}},")
    for codecell in nb.cells:
        if codecell.cell_type == 'code':
            compile(codecell.source, '<'+codecell.id+'>', 'exec')
project_manager.enqueue_write(handle, fix_definition)
lifecycle.mark_write_end('v020-exp-19')
print('構文修正、全既存コードセルの構文検査成功')

構文修正、全既存コードセルの構文検査成功


In [6]:
iteration1_request = '初回の散布図では二つの集まりがあり、全体相関の大半が群間差だった。これだけで個々の待ち時間を正確に予測できるとは言えないため、平均だけ・全体直線・短長群別平均・群別直線を同じ分割で交差検証し、残差と予測誤差を比較してください。またKaggleの公開メタデータで待ち時間の方向と単位を再確認し、確認できない定義は留保してください。'
iteration1_code = '''with phase('反復1：定義確認・予測誤差'):
    metadata_text = ''
    if selected['kind'] == 'kaggle':
        try:
            metadata_dir = data_dir / 'kaggle-metadata'
            metadata_dir.mkdir(exist_ok=True)
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                api.dataset_metadata(selected['owner_dataset_slug'], str(metadata_dir))
            metadata_file = metadata_dir / 'dataset-metadata.json'
            metadata_payload = json.loads(metadata_file.read_text(encoding='utf-8'))
            public_keys = ['id','title','subtitle','description','licenses']
            public_metadata = {k:metadata_payload[k] for k in public_keys if k in metadata_payload}
            metadata_text = str(public_metadata.get('description',''))
            print('Kaggle公開メタデータ:', json.dumps(public_metadata, ensure_ascii=False, indent=2))
        except Exception as exc:
            print('公開メタデータ取得失敗:', type(exc).__name__, '定義の留保を維持する')
    print('R辞書は Waiting time to next eruption と定義。ただしKaggleコピーの測定時刻・開始終了規約・元版との完全同一性は未確認。')
    def design(values, model, threshold=3.0):
        values = np.asarray(values)
        g = (values >= threshold).astype(float)
        if model == 'mean':
            return np.ones((len(values),1))
        if model == 'pooled_linear':
            return np.column_stack([np.ones(len(values)),values])
        if model == 'group_mean':
            return np.column_stack([np.ones(len(values)),g])
        if model == 'group_linear':
            return np.column_stack([np.ones(len(values)),values,g,values*g])
        raise ValueError('未知モデル:'+model)
    models = ['mean','pooled_linear','group_mean','group_linear']
    cv_rows = []
    x = df.duration.to_numpy()
    y = df.waiting.to_numpy()
    rng = np.random.default_rng(20261002)
    for repeat in range(20):
        permutation = rng.permutation(len(df))
        folds = np.array_split(permutation,5)
        for model in models:
            predictions = np.full(len(df), np.nan)
            for fold in folds:
                train = np.setdiff1d(np.arange(len(df)),fold)
                coefficients = np.linalg.lstsq(design(x[train],model),y[train],rcond=None)[0]
                predictions[fold] = design(x[fold],model) @ coefficients
            residual = y-predictions
            cv_rows.append({'repeat':repeat,'model':model,'rmse':float(np.sqrt(np.mean(residual**2))), 'mae':float(np.mean(np.abs(residual))), 'p90_absolute_error':float(np.quantile(np.abs(residual),.9))})
    cv_df = pd.DataFrame(cv_rows)
    cv_summary = cv_df.groupby('model').agg(RMSE_mean=('rmse','mean'), RMSE_split_sd=('rmse','std'), MAE_mean=('mae','mean'), P90_absolute_error_mean=('p90_absolute_error','mean')).reindex(models)
    display(cv_summary.round(5))
    paired_improvement = cv_df.pivot(index='repeat',columns='model',values='rmse')
    print('同じ分割でのRMSE改善（pooled_linear - group_linear）:', (paired_improvement.pooled_linear-paired_improvement.group_linear).describe().to_dict())
    fits = {}
    for model in models:
        coefficients = np.linalg.lstsq(design(x,model),y,rcond=None)[0]
        prediction = design(x,model) @ coefficients
        fits[model] = {'coefficients':coefficients, 'prediction':prediction, 'residual':y-prediction}
    print('全標本OLS係数（待ち時間分、噴出時間分）:', fits['pooled_linear']['coefficients'].tolist())
    print('群別直線係数[intercept,x,long_indicator,x*long_indicator]:',fits['group_linear']['coefficients'].tolist())
    pred_results = {'cv_summary':cv_summary.to_dict(), 'pooled_coefficients':fits['pooled_linear']['coefficients'].tolist(), 'group_linear_coefficients':fits['group_linear']['coefficients'].tolist(), 'paired_rmse_improvement':float((paired_improvement.pooled_linear-paired_improvement.group_linear).mean())}
    print('PREDICTION_RESULTS='+json.dumps(pred_results, ensure_ascii=False))
    display({'text/plain':'PREDICTION_RESULTS='+json.dumps(pred_results, ensure_ascii=False)},raw=True)
    (data_dir/'iteration-1-results.json').write_text(json.dumps(pred_results, ensure_ascii=False, indent=2),encoding='utf-8')
    fig,axes=plt.subplots(1,3,figsize=(14,4))
    grid=np.linspace(x.min(),x.max(),400)
    axes[0].scatter(x,y,s=12,alpha=.35,label='Observations')
    for model in models[1:]:
        axes[0].plot(grid,design(grid,model)@fits[model]['coefficients'],label=model)
    axes[0].set(title='Model fits (in-sample visualization)',xlabel='Duration (min)',ylabel='Waiting (min)')
    axes[0].legend(fontsize=7)
    for model in ['pooled_linear','group_linear']:
        axes[1].scatter(x,fits[model]['residual'],s=12,alpha=.4,label=model)
    axes[1].axhline(0,color='black',ls=':')
    axes[1].set(title='Residual structure',xlabel='Duration (min)',ylabel='Observed - fitted (min)')
    axes[1].legend(fontsize=7)
    axes[2].bar(np.arange(len(models)),cv_summary.RMSE_mean,yerr=cv_summary.RMSE_split_sd,capsize=4,label='Mean and SD across 20 splits')
    axes[2].set_xticks(np.arange(len(models)),models,rotation=30,ha='right',fontsize=8)
    axes[2].set(title='Repeated 5-fold CV (not future validation)',xlabel='Model',ylabel='RMSE (min)')
    axes[2].legend(fontsize=7)
    fig.tight_layout()
    emit_figure(fig,'反復1：モデル・残差・CV誤差')
    print('反復1の限界: 同一標本内の反復CV。分割間SDは信頼区間ではない。境界は探索的・固定で、最良境界の選択は行わない。残差約数分は平均の推定誤差ではなく個々の待ち時間の変動を含む。')
'''
compile(iteration1_code,'<iteration1>','exec')
lifecycle.mark_write_start('v020-exp-19')
def update_append(nb):
    definitions = next(c for c in nb.cells if c.id=='definitions')
    definitions.source=definitions.source.replace("'minutes' in dictionary_text", "'Eruption time in min' in dictionary_text and 'Waiting time to next eruption' in dictionary_text")
    summary = next(c for c in nb.cells if c.id=='summary')
    summary.source += "\n    display({'text/plain': f'PEARSON_R={pearson.statistic:.6f}; SPEARMAN_R={spearman.statistic:.6f}; WAIT_GAP={wait_gap:.6f}; BETWEEN_COV_FRACTION={between_fraction:.6f}; MAX_RELATIVE_DEVIATION={sensitivity_report.max_relative_deviation:.6f}'},raw=True)\n"
    nb.cells.extend([
        nbformat.v4.new_markdown_cell('## 初回結果の読み直し\n全体rは約0.901だが群内では約0.29–0.37。両変数に2峰、群の待ち時間差は約25.49分で、共分散の約94.6%が群間成分。境界/重複感度は安定。ただし強い相関は個々の予測精度を保証せず、図では群内の散らばりが残る。予測誤差と定義の確認が実質的な未解決点である。', id='initial-review'),
        nbformat.v4.new_markdown_cell('## 反復依頼履歴\n\n### 反復1：原文\n'+iteration1_request+'\n\n**選定理由:** 相関を予測能力と取り違えると結論が変わるため。\n**実行・証拠:** `iteration-1`セルの実出力、`iteration-1-results.json`と図。',id='iteration-1-request'),
        nbformat.v4.new_code_cell(iteration1_code,id='iteration-1',metadata={'chart':{'title':'Model fits / Residual structure / Repeated CV','xlabel':'Duration (min) / Model','ylabel':'Waiting / Residual / RMSE (min)','legend':'Observations / pooled_linear / group_mean / group_linear / split SD','missing_glyphs':[]}})])
project_manager.enqueue_write(handle,update_append)
lifecycle.mark_write_end('v020-exp-19')
print('初回の結果を読み直し、反復1依頼を原文で記録')

初回の結果を読み直し、反復1依頼を原文で記録


In [7]:
iteration2_request = '反復1では群別直線のRMSE改善が約0.28分と小さく、ランダム分割が行の独立性に依存している。ファイル順のラグ相関を参考として調べ、行順を保持した前方分割とブロック・ブートストラップで、正の関連と短長群の待ち時間差の不確実性を確認してください。3分の探索的境界を2.5〜3.5分へ変えた場合の予測モデルの優位性も検討し、物理的な時間順が未確認である限界を明記してください。'
iteration2_code = '''with phase('反復2：依存性・区間・予測仕様感度'):
    lag_diagnostics = {col:float(np.corrcoef(df[col].to_numpy()[:-1],df[col].to_numpy()[1:])[0,1]) for col in ['duration','waiting']}
    lag_diagnostics['group_linear_residual'] = float(np.corrcoef(fits['group_linear']['residual'][:-1],fits['group_linear']['residual'][1:])[0,1])
    print('ファイル順のラグ1相関（時間順とは断定しない）:',lag_diagnostics)
    def bootstrap_metrics(indices):
        bx,by = x[indices],y[indices]
        short = bx < 3.0
        if not short.any() or short.all():
            raise ValueError('bootstrap標本に片方の群しかない')
        return [float(np.corrcoef(bx,by)[0,1]),float(by[~short].mean()-by[short].mean())]
    bootstrap_rows = []
    bootstrap_distributions = {}
    for block_length in [1,5,10,20]:
        rng=np.random.default_rng(20261002+block_length)
        draws=[]
        for rep in range(1500):
            if block_length == 1:
                indices=rng.integers(0,len(df),len(df))
            else:
                starts=rng.integers(0,len(df),int(np.ceil(len(df)/block_length)))
                indices=((starts[:,None]+np.arange(block_length)[None,:])%len(df)).ravel()[:len(df)]
            draws.append(bootstrap_metrics(indices))
        values=np.asarray(draws)
        bootstrap_distributions[block_length]=values
        for j,metric in enumerate(['Pearson_r','long_minus_short_waiting_min']):
            lo,hi=np.quantile(values[:,j],[.025,.975])
            bootstrap_rows.append({'block_length':block_length,'metric':metric,'lower_95':float(lo),'upper_95':float(hi),'B':1500})
    bootstrap_summary=pd.DataFrame(bootstrap_rows)
    display(bootstrap_summary.round(6))
    forward_rows=[]
    test_blocks=np.array_split(np.arange(len(df)//2,len(df)),5)
    for model in models:
        errors=[]
        fold_rmse=[]
        for test in test_blocks:
            train=np.arange(test[0])
            beta=np.linalg.lstsq(design(x[train],model),y[train],rcond=None)[0]
            err=y[test]-design(x[test],model)@beta
            errors.extend(err.tolist())
            fold_rmse.append(float(np.sqrt(np.mean(err**2))))
        forward_rows.append({'model':model,'rmse':float(np.sqrt(np.mean(np.square(errors)))),'mae':float(np.mean(np.abs(errors))),'per_block_rmse':fold_rmse,'test_n':len(errors)})
    forward_summary=pd.DataFrame(forward_rows)
    display(forward_summary)
    spec_rows=[]
    def cv_improvement(threshold):
        errors={m:[] for m in ['pooled_linear','group_linear']}
        rng=np.random.default_rng(20261002)
        for repeat in range(10):
            folds=np.array_split(rng.permutation(len(df)),5)
            predictions={m:np.full(len(df),np.nan) for m in errors}
            for test in folds:
                train=np.setdiff1d(np.arange(len(df)),test)
                for model in errors:
                    beta=np.linalg.lstsq(design(x[train],model,threshold),y[train],rcond=None)[0]
                    predictions[model][test]=design(x[test],model,threshold)@beta
            for model in errors:
                errors[model].append(float(np.sqrt(np.mean((y-predictions[model])**2))))
        pooled=float(np.mean(errors['pooled_linear']))
        grouped=float(np.mean(errors['group_linear']))
        spec_rows.append({'threshold':threshold,'pooled_RMSE':pooled,'group_RMSE':grouped,'improvement_minutes':pooled-grouped})
        return pooled-grouped
    model_sensitivity=sensitivity.run_sensitivity(sensitivity.SensitivityPlan({'threshold':[3.0,2.5,2.75,3.25,3.5]},max_runs=5),cv_improvement,stability_tolerance=.20)
    display(pd.DataFrame(spec_rows).round(6))
    print('モデル優位性の感度:',json.dumps(asdict(model_sensitivity),ensure_ascii=False))
    iteration2_results={'lag1_file_order':lag_diagnostics,'bootstrap':bootstrap_rows,'forward_cv':forward_rows,'model_sensitivity':asdict(model_sensitivity),'threshold_cv':spec_rows}
    print('ITERATION2_RESULTS='+json.dumps(iteration2_results,ensure_ascii=False))
    display({'text/plain':'ITERATION2_RESULTS='+json.dumps(iteration2_results,ensure_ascii=False)},raw=True)
    (data_dir/'iteration-2-results.json').write_text(json.dumps(iteration2_results,ensure_ascii=False,indent=2),encoding='utf-8')
    fig,axes=plt.subplots(1,3,figsize=(13,4))
    for block_length,values in bootstrap_distributions.items():
        axes[0].hist(values[:,1],bins=35,density=True,histtype='step',label=f'block={block_length}')
    axes[0].set(title='Bootstrap: long - short waiting',xlabel='Mean gap (min)',ylabel='Density')
    axes[0].legend(fontsize=8)
    for model in ['pooled_linear','group_linear']:
        row=forward_summary.loc[forward_summary.model==model].iloc[0]
        axes[1].plot(np.arange(1,6),row.per_block_rmse,marker='o',label=model)
    axes[1].set(title='Forward splits in file order',xlabel='Test block (not verified time)',ylabel='RMSE (min)')
    axes[1].legend(fontsize=8)
    axes[2].plot([r['threshold'] for r in sorted(spec_rows,key=lambda r:r['threshold'])],[r['improvement_minutes'] for r in sorted(spec_rows,key=lambda r:r['threshold'])],marker='o',label='Pooled RMSE - group RMSE')
    axes[2].axhline(0,color='grey',ls=':')
    axes[2].set(title='Model advantage depends on threshold',xlabel='Duration threshold (min)',ylabel='RMSE improvement (min)')
    axes[2].legend(fontsize=8)
    fig.tight_layout()
    emit_figure(fig,'反復2：区間・前方分割・境界依存')
    print('残る限界: bootstrap区間は母集団抽出を保証しない参考的percentile区間。円環ブロックは末尾と先頭を接続する仮定を追加する。実観測順・日付なし。複数モデルを比較した探索的結果で、独立な最終テスト標本はない。')
'''
compile(iteration2_code,'<iteration2>','exec')
lifecycle.mark_write_start('v020-exp-19')
def append_second(nb):
    c=next(c for c in nb.cells if c.id=='iteration-1')
    c.source=c.source.replace("public_keys = ['id','title','subtitle','description','licenses']", "metadata_payload = metadata_payload.get('info', metadata_payload)\n            public_keys = ['id','title','subtitle','description','licenses']")
    nb.cells.extend([
        nbformat.v4.new_markdown_cell('## 反復1の結果と再評価\n**結果:** 反復5-fold CV（20分割）でRMSEは平均のみ13.63分、全体直線5.94分、群別平均5.96分、群別直線5.66分。群別直線の改善は約0.28分と小さく、90%絶対誤差は約9.10分。二群構造は有用だが個々の時刻を精密に決定しない。\n**証拠:** `iteration-1`実出力とモデル/残差/CV図。\n**残る限界:** 行の独立性とファイル順が未確認、境界依存、同一標本内のモデル比較。Kaggleメタデータのdescriptionは空でライセンスはCC0-1.0。R辞書は次の噴出までと定義するが、Kaggleコピーの測定規約・原版との完全同一性を追加確認できなかった。',id='iteration-1-review'),
        nbformat.v4.new_markdown_cell('### 反復2：原文\n'+iteration2_request+'\n\n**選定理由:** 予測改善が小さいため、独立性・境界という未検証の選択で結論が変わらないか確認する。\n**実行・証拠:** `iteration-2`セル、ブートストラップ1500回×4仕様・ファイル順前方分割・境界5仕様の出力。',id='iteration-2-request'),
        nbformat.v4.new_code_cell(iteration2_code,id='iteration-2',metadata={'chart':{'title':'Bootstrap gap / Forward splits / Threshold sensitivity','xlabel':'Mean gap / Test block / Duration threshold','ylabel':'Density / RMSE / RMSE improvement','legend':'block lengths 1,5,10,20 / pooled_linear / group_linear','missing_glyphs':[]}})])
project_manager.enqueue_write(handle,append_second)
lifecycle.mark_write_end('v020-exp-19')
print('反復2依頼を原文のまま記録')

反復2依頼を原文のまま記録


In [8]:
iteration3_request = '反復2で短長群の待ち時間差は再標本化でも大きく残ったが、群別直線の予測改善量は境界に敏感だった。Rデータ辞書が噴出時間の強い丸めを指摘しているため、噴出時間を±2.5秒および保守的に±5秒、整数の待ち時間を±0.5分揺らした感度分析を行い、相関・群間差・群所属・予測改善への影響を確認してください。これは測定誤差モデルの推定ではなく仮定に基づく摂動であると明記してください。'
iteration3_code = '''with phase('反復3：丸めに対する感度'):
    perturbation_rows=[]
    rng=np.random.default_rng(20261003)
    folds=np.array_split(np.random.default_rng(20261002).permutation(len(df)),5)
    for duration_halfwidth_seconds in [0.,2.5,5.]:
        for repeat in range(120):
            bx=x+rng.uniform(-duration_halfwidth_seconds/60,duration_halfwidth_seconds/60,len(df))
            by=y+(rng.uniform(-.5,.5,len(df)) if duration_halfwidth_seconds>0 else 0.)
            assigned=bx<3
            predictions={model:np.full(len(df),np.nan) for model in ['pooled_linear','group_linear']}
            for test in folds:
                train=np.setdiff1d(np.arange(len(df)),test)
                for model in predictions:
                    beta=np.linalg.lstsq(design(bx[train],model),by[train],rcond=None)[0]
                    predictions[model][test]=design(bx[test],model)@beta
            pooled_rmse=float(np.sqrt(np.mean((by-predictions['pooled_linear'])**2)))
            group_rmse=float(np.sqrt(np.mean((by-predictions['group_linear'])**2)))
            perturbation_rows.append({'duration_halfwidth_seconds':duration_halfwidth_seconds,'repeat':repeat,
                                     'r':float(np.corrcoef(bx,by)[0,1]),'gap':float(by[~assigned].mean()-by[assigned].mean()),
                                     'group_switches':int(np.sum(assigned!=(x<3))), 'pooled_RMSE':pooled_rmse,'group_RMSE':group_rmse,'improvement':pooled_rmse-group_rmse})
    perturbation_df=pd.DataFrame(perturbation_rows)
    jitter_summary=perturbation_df.groupby('duration_halfwidth_seconds').agg(r_mean=('r','mean'),r_min=('r','min'),r_max=('r','max'),gap_mean=('gap','mean'),gap_min=('gap','min'),gap_max=('gap','max'),switches_max=('group_switches','max'),improvement_mean=('improvement','mean'),improvement_min=('improvement','min'),improvement_max=('improvement','max'))
    display(jitter_summary.round(6))
    jitter_stability=sensitivity.run_sensitivity(sensitivity.SensitivityPlan({'halfwidth':[0.,2.5,5.]},max_runs=3),lambda halfwidth: float(jitter_summary.loc[halfwidth,'gap_mean']),stability_tolerance=.10)
    print('丸め摂動に対する群間差の感度:',json.dumps(asdict(jitter_stability),ensure_ascii=False))
    iteration3_results={'summary':jitter_summary.to_dict(),'gap_stability':asdict(jitter_stability),'n_runs':len(perturbation_df),'assumption':'一様摂動は測定誤差分布の推定ではない。±5秒は保守的なシナリオ。境界は3分固定、CV分割は1セット固定。'}
    display({'text/plain':'ITERATION3_RESULTS='+json.dumps(iteration3_results,ensure_ascii=False)},raw=True)
    (data_dir/'iteration-3-results.json').write_text(json.dumps(iteration3_results,ensure_ascii=False,indent=2),encoding='utf-8')
    perturbation_df.to_csv(data_dir/'rounding-sensitivity.csv',index=False)
    print('反復3の限界:',iteration3_results['assumption'])
'''
probe_code = '''with phase('Jupytermind改善候補の再現'):
    from dataclasses import replace
    probe_handle = replace(handle, notebook_path=data_dir/'visual-audit-probe.ipynb')
    lifecycle.mark_write_start(RUN_ID)
    try:
        project_manager.ensure_notebook(probe_handle)
        project_manager.enqueue_write(probe_handle, lambda nb: nb.cells.clear())
        probe_png = visualization.render_chart(df, kind='scatter', x='duration', y='waiting', title='Audit probe', xlabel='Duration (min)', ylabel='Waiting (min)')
        visualization.record_chart(probe_handle, "display(visualization.build_image_output(probe_png).data, raw=True)", probe_png)
    finally:
        lifecycle.mark_write_end(RUN_ID)
    probe_nb=nbformat.read(probe_handle.notebook_path,as_version=4)
    probe_audit=notebook_audit.audit_notebook(probe_handle.notebook_path,visual_audit=True)
    probe_result={'module':'ai_data_scientist.visualization.record_chart / ai_data_scientist.notebook_audit.audit_visual_outputs',
                  'chart_metadata_present':bool(probe_nb.cells[0].metadata.get('chart')),
                  'audit_ok_without_chart_metadata':probe_audit.ok,
                  'visual_findings':[asdict(f) for f in probe_audit.visual_findings],
                  'expected':'record_chartがタイトル・軸・凡例・グリフメタデータを保存する、または監査が欠落を指摘する',
                  'impact':'visual_audit=Trueが未記録の図表属性を見逃し得る。今回の主Notebookは手動でmetadata.chartを記録し欠落グリフを実描画で確認。',
                  'workaround':'enqueue_writeで明示的chartメタデータ、描画警告の捕捉、全図の目視確認',
                  'classification':'図表記録/監査の機能不足（Kaggle API、Copilot CLI、ベンチマークランナーとは独立）'}
    print('JUPYTERMIND_PROBE='+json.dumps(probe_result,ensure_ascii=False))
    display({'text/plain':'JUPYTERMIND_PROBE='+json.dumps(probe_result,ensure_ascii=False)},raw=True)
    (data_dir/'jupytermind-improvement-probe.json').write_text(json.dumps(probe_result,ensure_ascii=False,indent=2),encoding='utf-8')
    probe_handle.notebook_path.unlink()
    print('診断用一時Notebookを削除。再現ログJSONは保持。')
'''
for name,code in [('iteration3',iteration3_code),('probe',probe_code)]:
    compile(code,'<'+name+'>','exec')
lifecycle.mark_write_start('v020-exp-19')
def append_third(nb):
    nb.cells.extend([
        nbformat.v4.new_markdown_cell('## 反復2の結果と再評価\n**結果:** IIDとブロック長5/10/20の参考95%区間はrがおよそ0.882–0.918、待ち時間差がおよそ23.84–27.09分で、関連と差の符号は保たれる。ファイル順前方分割でもRMSEは全体直線5.96分、群別直線5.73分。ラグ1相関は噴出時間−0.550、待ち時間−0.540、群別直線残差−0.223で、行順を無視した独立性は確定できない。\n**感度:** 2.5–3.5分で予測改善は0.064–0.290分、stable=False、最大相対偏差0.779。群間差の頑健性と予測改善量の頑健性は異なる。\n**証拠:** `iteration-2`出力と図、`iteration-2-results.json`。\n**残る限界:** ファイル順を真の時系列と同一視しない。bootstrapは代表性を保証しない。測定値の丸めが小さな予測改善を左右する可能性を最後に確認する。',id='iteration-2-review'),
        nbformat.v4.new_markdown_cell('### 反復3：原文\n'+iteration3_request+'\n\n**選定理由:** 出典に記載された丸めが、重複ペアと小さなモデル改善の解釈に影響し得るため。\n**実行・証拠:** `iteration-3`セル、120回×3仕様の固定乱数感度分析。',id='iteration-3-request'),
        nbformat.v4.new_code_cell(iteration3_code,id='iteration-3'),
        nbformat.v4.new_markdown_cell('## Jupytermind改善候補\n分類は図表記録/監査の機能不足。再現条件・期待・実際・影響・回避策は下の`jupytermind-probe`実出力と`data/jupytermind-improvement-probe.json`に記録する。`record_chart`で保存した図にchartメタデータがなく、visual_audit=Trueでも欠落が検出されないことを最小の独立Notebookで確認する。今回の図表は手動メタデータ・グリフ警告検査で回避。Kaggle SDK引数誤り、分析コードの構文誤りは修正済みで、これとは別の問題。コード変更やIssue登録は本依頼の対象外。',id='improvement-candidates'),
        nbformat.v4.new_code_cell(probe_code,id='jupytermind-probe')])
project_manager.enqueue_write(handle,append_third)
lifecycle.mark_write_end('v020-exp-19')
print('反復3と独立した機能不足診断を記録')

反復3と独立した機能不足診断を記録


In [9]:
modules_note = '''## 使用したJupytermindモジュール
直接import・呼び出しを行ったものだけを記載する。`ai_data_scientist`パッケージ名は省略せず、下表の関数/クラスだけを使用済みとする。間接呼出しは含めない。

| モジュール名 | 直接呼び出した関数・クラス・メソッド | 使用目的 |
|---|---|---|
| ai_data_scientist.project_manager | resolve_project, ensure_notebook, ensure_data_dir, enqueue_write | プロジェクト解決、データ/Notebook作成、直列化書込み |
| ai_data_scientist.language_router | detect_language | 日本語の判定 |
| ai_data_scientist.lifecycle | register_run, mark_execution_start/end, mark_write_start/end, is_cancel_requested, mark_completed, mark_failed, get_run_status, wait_for_quiescence | 登録、実行/書込み記録、終了/静止確認 |
| ai_data_scientist.ingestion | SourceSpec, ingest | 取得CSVの読込み、上限検査 |
| ai_data_scientist.cleaning | clean_dataset | 欠損除去の影響記録。除去0行 |
| ai_data_scientist.eda | explore | 型、欠損、統計要約 |
| ai_data_scientist.stats_analysis | correlation | Pearson係数と日本語の解釈 |
| ai_data_scientist.visualization | render_chart, build_image_output, record_chart | 日本語図・画像MIME、record_chartは改善候補の再現だけ |
| ai_data_scientist.data_definition | FieldValue, build_manifest, DataDefinitionManifest.unresolved_fields | 定義の確度、未確認項目。inferredは別途抽出 |
| ai_data_scientist.analysis_assumptions | Assumption, AnalysisAssumptionManifest, check_manifest | 非因果の範囲・仮定・警告の記録 |
| ai_data_scientist.data_quality | detect_anomalies | 広い意味的範囲とnot_null検査 |
| ai_data_scientist.sensitivity | SensitivityPlan, run_sensitivity | 境界・重複・予測仕様・丸め感度 |
| ai_data_scientist.insight_engine | extract_cited_value, record_insight | 実出力から根拠値を抽出し証拠つき結論を生成 |
| ai_data_scientist.notebook_audit | audit_notebook | visual_audit=Trueで実行・証拠・図表を監査 |

`notebook_audit.audit_visual_outputs`はaudit_notebookからの間接利用なので直接使用済みと数えない。`dataset_validation.compare_datasets`と`data_quality.validate_anomalies`は独立参照標本がなく適用外。単一正規分布のz-score異常検知は二峰混合を異常扱いし得るため採用しない。ML拡張のクラスタリング/AutoMLは使用しない。

### Jupyter MCP実行経路とgatewayの適用範囲
分析・APIコードの実行は実際のJupyter MCP execute_cellだけで行った。Notebookセルの手動追加・編集は別のJupyter MCP制御Notebookからenqueue_writeで直列化し、実行出力はMCPが実際のカーネルexecution_countとともに保存する。Pythonの同期MCPClientはCLIツール環境に提供されていないため、mcp_gateway.run_and_recordは未使用。擬似clientやexec/外部スクリプトで使用したことにはしない。これはホスト側の接続インターフェース制約で、Jupytermind不具合とは分類しない。
'''
final_code = '''with phase('結論の証拠更新・最終監査'):
    current_nb = nbformat.read(handle.notebook_path,as_version=4)
    evidence_cells = {c.id:c for c in current_nb.cells if c.cell_type=='code'}
    def evidence_output(cell_id):
        cell=evidence_cells[cell_id]
        return {'output':'\\n'.join(str(o.get('data',{}).get('text/plain','')) for o in cell.outputs)},cell.execution_count
    queued_write(lambda nb: setattr(nb,'cells',[c for c in nb.cells if not c.metadata.get('generated_insight')]))
    claims = [
        ('summary',r'PEARSON_R=([0-9.]+)', '**全体の関連:** Pearson r={value}、Spearmanは約0.778で強い正の関連。Jupytermindの解釈は「強い正の相関が見られる」。ただし分布は一つの正規的な集まりではない。KDEでは噴出時間が約2.0/4.36分、待ち時間が約53.7/79.9分の2峰となり、Scott帯域の0.5–1.25倍でも2峰が残った。因果効果を示さない。', 'associational'),
        ('summary',r'WAIT_GAP=([0-9.]+)', '**条件別の分布:** 3分未満97観測（35.7%）では平均噴出2.038分・待ち54.495分、3分以上175観測（64.3%）では4.291分・79.989分。待ち時間の平均差は{value}分。全体平均70.897分を典型とすると、二つの主要パターンを見落とす。群内SDは約5.84/5.99分、待ち時間の64–71分付近には重なりがあり完全分離ではない。', 'descriptive'),
        ('summary',r'BETWEEN_COV_FRACTION=([0-9.]+)', '**単純相関を超える構造:** 全共分散に占める群間成分の比率は{value}（約94.6%）。群内rはshort約0.290、long約0.373まで小さくなる。強い全体相関の多くは短長パターンの差に対応する。ただし噴出時間で群を切った範囲制限もあるため「群内に関係がない」「Simpsonの逆転が起きた」とは言わない。比率は因果寄与率でもR2でもない。', 'associational'),
        ('summary',r'MAX_RELATIVE_DEVIATION=([0-9.]+)', '**群間差の感度:** 2.5–3.5分の境界×重複保持/除去の10仕様で差は24.37–25.49分。stable=True、最大相対偏差={value}（約4.42%、事前に置いた許容10%内）。16件の同一ペアは丸めや別イベントの可能性があるため本分析では保持した。欠損/意味的異常は0、欠損処理の除去も0行。', 'descriptive'),
        ('iteration-1',r'"RMSE_mean":.*?"group_linear": ([0-9.]+)', '**予測の限界（反復1）:** 20回の反復5-fold CVで群別直線の平均RMSE={value}分。平均のみ13.63分、全体直線5.94分、群別平均5.96分に対し改善は約0.28分。群別直線でも90%絶対誤差は約9.10分。群所属は重要だが「長い噴出なら何分後かが精密に分かる」とは結論しない。比較は同一標本内であり、将来への保証ではない。', 'associational'),
        ('iteration-2',r'"block_length": 20, "metric": "long_minus_short_waiting_min", "lower_95": ([0-9.]+)', '**再標本化と依存性（反復2）:** ブロック長20の待ち時間平均差の参考95%percentile区間は下限{value}分、上限約27.09分。IID/他のブロック長でも差とrの符号は保たれる。ファイル順のラグ1相関は噴出−0.550/待ち−0.540で、時間順未確認のため物理的な交互噴出や周期の証明とはしない。前方分割RMSEは全体5.96分、群別5.73分。', 'associational'),
        ('iteration-2',r'"max_relative_deviation": ([0-9.]+)', '**予測優位性は限定的（反復2）:** 境界5仕様で群別直線のRMSE改善量は0.064–0.290分。stable=False、最大相対偏差={value}（約77.9%、許容20%超）。すべての検討境界で改善は正だったが、小さな改善量を境界に対して頑健とは呼ばない。差の頑健性と予測改善量の頑健性を区別する。', 'associational'),
        ('iteration-3',r'"max_relative_deviation": ([0-9.]+)', '**丸め感度と停止判断（反復3）:** 360回の仮定的摂動で平均待ち時間差のstable=True、最大相対偏差={value}。噴出±5秒/待ち±0.5分の範囲でもrは約0.8965–0.9028、差は約25.34–25.58分、群の移動は最大1観測だった。固定分割の小さなモデル改善も正だが、測定誤差分布の推定ではない。二群の記述・限定的予測という主結論を変える追加分析は現データでは残らないため3回で停止。観測日時・本来の対応規約・独立外部標本がなければ残る限界は解消できない。', 'descriptive')]
    for cell_id,pattern,template,claim_type in claims:
        result,execution_count=evidence_output(cell_id)
        cited=insight_engine.extract_cited_value(result,pattern)
        lifecycle.mark_write_start(RUN_ID)
        log_event('insight','write_start')
        try:
            insight_engine.record_insight(handle,template.format(value=cited),execution_count,cited,claim_type,language='ja')
        finally:
            lifecycle.mark_write_end(RUN_ID)
            log_event('insight','write_end')
        def tag_latest(nb):
            nb.cells[-1].metadata['generated_insight']=True
        queued_write(tag_latest)
    def reorder_and_metadata(nb):
        tail=next(c for c in nb.cells if c.id=='final-audit')
        nb.cells=[c for c in nb.cells if c.id!='final-audit']+[tail]
        nb.metadata['run_id']=RUN_ID
        nb.metadata['provenance']=provenance
        nb.metadata['data_definition_manifest']=asdict(definitions)
        nb.metadata['analysis_assumptions_manifest']=asdict(assumptions)
        nb.metadata['chart_render_checks']=chart_checks
        nb.metadata['execution_route']='Jupyter MCP only; clean-kernel top-to-bottom execute_cell'
    queued_write(reorder_and_metadata)
    final_audit=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    print('notebook_audit(visual_audit=True):',json.dumps({**asdict(final_audit),'ok':final_audit.ok},ensure_ascii=False,indent=2))
    if not final_audit.ok:
        raise RuntimeError('Notebook監査不合格。未実行・エラー・証拠・図表を修正する必要がある')
    if any(c['missing_glyphs'] for c in chart_checks):
        raise RuntimeError('グリフ警告が残っている')
    print('最終的な前提finding:',[asdict(f) for f in analysis_assumptions.check_manifest(assumptions)])
    print('定義の留保: 分の単位・意味はR faithful辞書とCSV列名/値に準拠した推定。Kaggle公開descriptionは空。原版との完全同一性、噴出間隔の測定開始/終了規約、実観測順、日時、現在への代表性は未確認。因果・運用予測は行わない。')
lifecycle.mark_completed(RUN_ID)
quiescent=lifecycle.wait_for_quiescence(RUN_ID,timeout_s=5)
status=asdict(quiescent)
print('ライフサイクル終了:',json.dumps(status,ensure_ascii=False))
if status['state']!='completed' or any(status[k] for k in ['active_cell_executions','pending_notebook_writes','locks_held']):
    raise RuntimeError('終了状態がcompleted/静止ではない')
queued_write(lambda nb: nb.metadata.update({'lifecycle_final':status,'phase_log':phase_log,'audit_in_final_cell':{**asdict(final_audit),'ok':final_audit.ok}}))
(data_dir/'lifecycle.json').write_text(json.dumps({'final':status,'phase_log':phase_log},ensure_ascii=False,indent=2),encoding='utf-8')
print('保存先:',handle.notebook_path)
'''
compile(final_code,'<final>','exec')
lifecycle.mark_write_start('v020-exp-19')
def complete_structure(nb):
    definitions=next(c for c in nb.cells if c.id=='definitions')
    definitions.source=definitions.source.replace("'verified' if dictionary_verified else 'inferred'", "'inferred'")
    log=next(c for c in nb.cells if c.id=='sdk-log')
    log.source += '\n\n初回の暫定フォールバック記録: URL `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/geyser.csv`、取得日時UTC `2026-10-01T18:49:09.453873+00:00`、ファイル `geyser-fallback.csv`、SHA-256 `ce8f6bd15967c9a3dee345aaf268f6b92623abb1e1d313e04d79b720aa6b8bd6`。理由は未対応SDK引数による検索失敗。公開CSVとKaggle掲載版の完全な同一性は保証されない。SDK呼出しを修正後にKaggle取得成功し、最終分析にはKaggle CSVのみを使用。暫定CSVは整理時に削除する。\n\n定義セルの初回に余分な閉じ括弧のSyntaxErrorが発生したが修正・構文検査済み。HTMLのminタグでminutesが分割されるため辞書の文字列判定も修正した。メタデータのinfoネストを確認して公開項目の抽出を修正した。いずれも分析コード側の修正であり、Jupytermindの障害とはしない。'
    nb.cells.extend([
        nbformat.v4.new_markdown_cell('## 反復3の結果・反復停止・解釈上の留保\n**結果:** rは摂動全体で約0.8965–0.9028、待ち時間差は約25.34–25.58分、群移動は最大1件。群間差の最大相対偏差は約0.0466%でstable=True。固定分割の予測改善は約0.293–0.387分の小さな範囲。\n**証拠:** `iteration-3`セル、`iteration-3-results.json`、`rounding-sensitivity.csv`。\n**残る限界:** 想定した一様摂動であり実測誤差分布ではない。測定規約・観測時刻・新しい独立標本はこのCSVから復元できない。3回で、結論を変え得る現データ内の懸念を検討し終えたため停止する。未確認情報を補う再調査なしに追加検定を増やす価値は低い。\n\n分の単位/意味はR公式faithful辞書に準拠した推定としてmanifestにinferredを残す。Kaggle公開descriptionが空なので、原版との完全同一性や次回への方向対応を確定としない。check_manifestの対応方向・行独立性の2警告は隠さず残し、因果効果・現代の運用予測を主張しない。',id='iteration-3-review'),
        nbformat.v4.new_markdown_cell(modules_note,id='modules-used'),
        nbformat.v4.new_markdown_cell('## 最終的な日本語の結論と根拠\n以下の結論は、最後のセルが実行済みセルの実出力から値を抽出して生成する。根拠manifestは再実行時のexecution_countへ自動更新し、丸めた値を手入力の根拠にはしない。3分境界は探索的な区分であり、自然法則や因果的な体制変化の証明ではない。',id='conclusions-heading'),
        nbformat.v4.new_code_cell(final_code,id='final-audit')])
    for c in nb.cells:
        if c.cell_type=='code':
            compile(c.source,'<'+c.id+'>','exec')
project_manager.enqueue_write(handle,complete_structure)
lifecycle.mark_write_end('v020-exp-19')
print('最終セルとモジュール使用記録を追加。クリーンカーネルで全10コードセルを順に再実行する。')

最終セルとモジュール使用記録を追加。クリーンカーネルで全10コードセルを順に再実行する。
